# ISDF Coulomb Kernel $W$ — Tensor Train Decompositions

Explores QTT/MPS representations of $W[Q, \mu, \nu]$ (shape $64 \times 130 \times 130$) from a $4\times4\times4$ $k$-mesh diamond ISDF calculation.

## Setup

In [3]:
import sys
import os
import pickle
import signal
signal.signal(signal.SIGINT, signal.SIG_DFL) 

from pyscf.pbc.df.df import CDERIArray
from pyscf import lib
from pyscf.pbc import gto, scf, df
import numpy as np

import pickle
import h5py

In [4]:
nk = 4
cisdf = 5
norm_ratio = 0.1

basis = "gth-dzvp"
ke_cutoff = 40.0
kmesh = np.array([nk, nk, nk])
klabel = f"{kmesh[0]}x{kmesh[1]}x{kmesh[2]}"
scf_pkl = f"data/SCF_diamond_{klabel}_{basis}_ke{ke_cutoff}.pkl"
isdf_chk = f"data/ISDFopt_diamond_{klabel}_{basis}_c{cisdf}_norm{norm_ratio}.chk"

with open(scf_pkl, "rb") as f:
    mf = pickle.load(f)

cell = mf.cell
kpts = cell.make_kpts(kmesh)

C = np.asarray(mf.mo_coeff)
nkpts, nao, nmo = C.shape
nocc = cell.nelectron // 2
nvir = nmo - nocc
Cocc = np.array(C[:, :, :nocc], order="C", copy=True)
Cvir = np.array(C[:, :, nocc:], order="C", copy=True)

with h5py.File(isdf_chk, "r") as f:
    X = np.asarray(f["inpv_kpt"])
    W = np.asarray(f["coul_kpt"])

Xo = X @ Cocc
Xv = X @ Cvir

print("isdf_chk =", isdf_chk)
print("nkpts =", nkpts, "nao =", nao, "nocc =", nocc, "nvir =", nvir)
print("X.shape =", X.shape, "W.shape =", W.shape)

isdf_chk = data/ISDFopt_diamond_4x4x4_gth-dzvp_c5_norm0.1.chk
nkpts = 64 nao = 26 nocc = 4 nvir = 22
X.shape = (64, 130, 26) W.shape = (64, 130, 130)


## 1. QTT on $Q$ Index (Open $\mu$/$\nu$ Boundaries)

Decompose the $Q$ index into $n$ sites of physical dimension `phys_dim`, keeping $\mu$ and $\nu$ as open boundary indices:

$$W[Q,\mu,\nu] \approx \sum_{\alpha} A_0[\mu,\, s_0,\, \alpha_0]\; A_1[\alpha_0,\, s_1,\, \alpha_1] \;\cdots\; A_{n-1}[\alpha_{n-2},\, s_{n-1},\, \nu]$$

where $Q$ is recovered from the site indices $(s_0, s_1, \ldots)$ in base `phys_dim`.

In [9]:
import quimb.tensor as qtn
import math

# QTT of W[Q, mu, nu] with mu (left) and nu (right) as open boundary indices.
# Set phys_dim to control the base: 2 = binary QTT, 4 = base-4, etc.
#
# Each site is a 3-tensor with axes (l, r, p):
#   l = left bond dimension
#   r = right bond dimension
#   p = physical dimension (= phys_dim)
# Boundary sites absorb mu (left) or nu (right) in place of l or r.
# quimb stores cores as (l, p, r); we use core[:, p_val, :] to get the (l, r) matrix at fixed p.

phys_dim = nk   # ← change to 4, 8, ... for a different base

Q, mu_dim, nu_dim = W.shape
n_sites = math.ceil(math.log(Q, phys_dim))
Q_pad   = phys_dim ** n_sites

W_pad = np.zeros((Q_pad, mu_dim, nu_dim), dtype=W.dtype)
W_pad[:Q] = W
arr = W_pad.transpose(1, 0, 2).reshape([mu_dim] + [phys_dim]*n_sites + [nu_dim])

T = qtn.Tensor(arr, inds=['mu'] + [f's{k}' for k in range(n_sites)] + ['nu'])
cores = []
for k in range(n_sites - 1):
    left_inds = (['mu'] if k == 0 else [f'bond{k-1}']) + [f's{k}']
    T_left, T = T.split(left_inds, cutoff=3E-3, cutoff_mode='rel', bond_ind=f'bond{k}')
    cores.append(T_left)
cores.append(T)

# Reconstruct W from cores and compute errors.
specnorm = lambda M: np.linalg.svd(M, compute_uv=False)[0]

W_rec = np.zeros_like(W)
for q in range(Q):
    # Convert q to n_sites digits in base phys_dim, MSB first (matches reshape order)
    tmp, digits = q, []
    for _ in range(n_sites):
        digits.append(tmp % phys_dim)
        tmp //= phys_dim
    digits.reverse()
    # Contract: for each site, pick the (l, r) matrix at physical index p=digits[k]
    M = cores[0].data[:, digits[0], :]          # (mu_dim, r0)
    for k in range(1, n_sites):
        M = M @ cores[k].data[:, digits[k], :]  # → (mu_dim, nu_dim)
    W_rec[q] = M

# ── Errors ────────────────────────────────────────────────────────────────────
# Relative Frobenius error
frob_err = np.linalg.norm(W_rec - W) / np.linalg.norm(W)

# Relative 2-norm error: max_Q ||W_rec[q] - W[q]||_2 / max_Q ||W[q]||_2
err2_per_q = np.array([specnorm(W_rec[q] - W[q]) for q in range(Q)])
ref2_per_q = np.array([specnorm(W[q])             for q in range(Q)])
spec_err   = err2_per_q.max() / ref2_per_q.max()

# ── Norms ─────────────────────────────────────────────────────────────────────
# Norm 1 — original W: max_Q ||W[Q, :, :]||_2  (operator norm, max over Q blocks)
w_opnorm = ref2_per_q.max()

# Norm 2 — QTT bound: prod_k  max_p ||core_k[:, p, :]||_2
#   For each site, view core[:, p, :] as an (l, r) matrix at fixed physical index p,
#   take the max operator norm over p, then multiply across all sites.
def site_max_opnorm(core_data):
    return max(specnorm(core_data[:, p, :]) for p in range(core_data.shape[1]))

norm_qtt_bound = math.prod(site_max_opnorm(c.data) for c in cores)

# ── Summary ───────────────────────────────────────────────────────────────────
N_orig = W.size
N_comp = sum(c.data.size for c in cores)
saving = (1 - N_comp / N_orig) * 100

print(f"W.shape={W.shape}, Q padded to {phys_dim}^{n_sites}={Q_pad}  (phys_dim={phys_dim})\n")
for i, c in enumerate(cores):
    print(f"core[{i}]: shape=(l={c.data.shape[0]}, r={c.data.shape[2]}, p={c.data.shape[1]}), inds={c.inds}")
print(f"\nBond dims:  {[c.data.shape[-1] for c in cores[:-1]]}")
print(f"N_orig={N_orig:,}  N_comp={N_comp:,}  saving={saving:.1f}%")
print(f"\nRel. Frobenius error:           {frob_err:.3e}")
print(f"Rel. 2-norm error (max over Q): {spec_err:.3e}")
print(f"\n||W||_op  = max_Q ||W[Q]||_2:                    {w_opnorm:.4f}")
print(f"QTT bound = prod_k max_p ||core_k(p)||_op:       {norm_qtt_bound:.4f}")

W.shape=(64, 130, 130), Q padded to 4^3=64  (phys_dim=4)

core[0]: shape=(l=130, r=255, p=4), inds=('mu', 's0', 'bond0')
core[1]: shape=(l=255, r=421, p=4), inds=('bond0', 's1', 'bond1')
core[2]: shape=(l=421, r=130, p=4), inds=('bond1', 's2', 'nu')

Bond dims:  [255, 421]
N_orig=1,081,600  N_comp=780,940  saving=27.8%

Rel. Frobenius error:           1.109e-02
Rel. 2-norm error (max over Q): 3.203e-03

||W||_op  = max_Q ||W[Q]||_2:                    1.3889
QTT bound = prod_k max_p ||core_k(p)||_op:       2.4603


## 2. Per-$(\mu,\nu)$ QTT (No Open Bonds)

For each fixed $(\mu, \nu)$, decompose the length-$Q$ vector $W[:, \mu, \nu]$ into a binary MPS with no open bonds. This gives bond dimensions as a function of $(\mu, \nu)$, showing how entangled the $Q$-dependence is for each matrix element.

In [33]:
import quimb.tensor as qtn

# For each fixed (mu, nu), decompose W[:, mu, nu] into QTT with no open bonds.
# Uses quimb MatrixProductState: core[0]:(2,r1), core[k]:(r_{k-1},2,r_k), core[-1]:(r_{n-2},2)

Q, mu_dim, nu_dim = W.shape
n_bits = Q.bit_length() - 1 if Q & (Q - 1) == 0 else Q.bit_length()
Q_pad, n_bonds = 1 << n_bits, n_bits - 1

all_bd = np.zeros((mu_dim, nu_dim, n_bonds), dtype=int)
for mu_i in range(mu_dim):
    for nu_i in range(nu_dim):
        v = np.zeros(Q_pad, dtype=W.dtype)
        v[:Q] = W[:, mu_i, nu_i]
        mps = qtn.MatrixProductState.from_dense(v, dims=[2]*n_bits)
        mps.compress(cutoff=1e-10, cutoff_mode='rel')
        all_bd[mu_i, nu_i] = [mps.bond_size(i, i+1) for i in range(n_bonds)]

print(f"QTT bond dims for each W[:, mu, nu] (W.shape={W.shape})\n")
print(f"{'bond':>6}  {'max poss':>8}  {'min':>4}  {'mean':>6}  {'max':>4}")
for b in range(n_bonds):
    mp = min(2**(b+1), 2**(n_bits-b-1))
    bd = all_bd[:, :, b]
    print(f"  {b}–{b+1}    {mp:>8}  {bd.min():>4}  {bd.mean():>6.2f}  {bd.max():>4}")

np.set_printoptions(linewidth=300, threshold=100_000)
for b in range(n_bonds):
    print(f"\nbond {b}–{b+1}  (max poss={min(2**(b+1), 2**(n_bits-b-1))}):")
    print(all_bd[:, :, b])


SystemError: CPUDispatcher(<function svd_truncated_numba at 0x7f03cfc36a20>) returned a result with an exception set